# 1. Load Iris and make one shared split

Airflow runs this notebook first. Both model notebooks will read the same saved training and test rows.

## Execution parameters
Airflow injects these values into the tagged parameter cell. For manual execution, keep the defaults and run the five notebooks in order.

In [ ]:
# Airflow supplies these values. Defaults also let you run the notebooks manually.
project_root = ""
run_dir = ""
workflow_run_id = ""
experiment_name = "iris-pipeline-comparison"
registered_model_name = "iris-pipeline-classifier"

## Locate files and connect to MLflow
SQLite holds experiment and registry records. This run folder holds data and small result files shared between notebook tasks.

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

# Find the project even when VS Code starts in a notebook subfolder.
project_path = Path(project_root).resolve() if project_root else Path.cwd().resolve()
while not (project_path / "requirements.txt").is_file():
    if project_path == project_path.parent:
        raise RuntimeError("Open the notebook from inside the workshop project.")
    project_path = project_path.parent

# Airflow gives each execution its own folder. Manual runs use outputs/manual.
run_path = Path(run_dir) if run_dir else project_path / "outputs" / "manual"
run_path.mkdir(parents=True, exist_ok=True)

# All five notebooks use the same tracking database and registry.
tracking_uri = "sqlite:///" + (project_path / "mlflow.db").as_posix()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient()
print("Run folder:", run_path)

## Load the built-in dataset
Iris has 150 rows, four measurements, and three flower classes. No download or feature engineering is needed.

In [ ]:
from uuid import uuid4
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X = iris.data.astype("float64")
y = iris.target
print("Loaded Iris:", len(X), "rows and", X.shape[1], "features")
X.head()

## Split and save
Keep 20% for testing. Seed 42 makes the split repeatable; stratification keeps class proportions. The execution ID lets later notebooks reject results from another execution.

In [ ]:
random_state = 42
test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)
execution_id = workflow_run_id or uuid4().hex
split = {
    "execution_id": execution_id,
    "random_state": random_state,
    "test_size": test_size,
    "feature_names": list(X.columns),
    "X_train": X_train.values.tolist(),
    "X_test": X_test.values.tolist(),
    "y_train": y_train.tolist(),
    "y_test": y_test.tolist(),
}
(run_path / "split.json").write_text(json.dumps(split, indent=2), encoding="utf-8")
print(f"Split Iris: {len(X_train)} training rows, {len(X_test)} test rows.")

## Prepare the experiment
An experiment groups related training attempts. Create it once, with a local artifact folder; the two training notebooks will reuse it.

In [ ]:
if client.get_experiment_by_name(experiment_name) is None:
    client.create_experiment(
        experiment_name,
        artifact_location=(project_path / "mlartifacts" / "pipeline").as_uri(),
    )
experiment = mlflow.set_experiment(experiment_name)
print("Experiment:", experiment.name)
print("Saved data:", run_path / "split.json")

Next, Airflow can start both training notebooks independently. Setup has created no training runs yet.